# Latent Lens, step 7: supervised baseline with each camera held out

The step-6 notebook ran the supervised runs before its script file existed, so they were skipped. This notebook fixes that order.

**Before running:** Add Input -> **Notebook Output**: all earlier notebooks (Latent Lens 0 to 6). Settings -> GPU, Internet On. Takes about 40 minutes (9 short runs).


In [ ]:
!pip -q install timm
import torch
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available())

In [ ]:
%%writefile latentlens_train.py
#!/usr/bin/env python
"""Latent Lens, step 1: camera-aware contrastive fine-tuning + evaluation.

Needs the cache made by step 0 (images_224.npy + meta.csv). On Kaggle attach the output of your
"Latent Lens 0" notebook as an input; the script finds the cache by itself.

  !python latentlens_train.py --variant frozen              # B1: frozen DINOv2-S, same mask, no training
  !python latentlens_train.py --variant b3                  # B3: plain SimCLR, warm-started from DINOv2-S
  !python latentlens_train.py --variant b4                  # B4: Latent Lens (cross-camera positives,
                                                            #     false-negative mask, Fourier style swap)
  !python latentlens_train.py --variant b4 --hold cam1      # same, but cam1 is never seen in training
  !python latentlens_train.py --stage reeval                # re-score saved features after per-camera centering
  !python latentlens_train.py --stage report                # table of every finished run (mean +- std over seeds)

Variants: frozen, b3, b4, and ablations b4_no_nn (= b3 + fn + fmix), b4_no_fmix, b4_no_fn, plus the ladder b3_fn, b3_fmix.
Every run writes results/<run>.json. Smoke test: add --smoke (1 epoch, 3 steps).

Honest protocol, fixed before seeing results:
  * Contrastive training uses ONLY unlabeled RFMiD *train* images (and never the --hold camera).
  * Linear probes are fit on RFMiD train, scored on RFMiD val / test, on the held-out camera and on IDRiD.
  * Few-shot support sets are drawn with fixed seeds, so every variant sees the same support sets.
  * A circular retina mask is applied to every input of every model (also the frozen baseline).
"""
import os, re, sys, json, time, glob, random, argparse
import numpy as np
import pandas as pd

WORK = os.environ.get('LL_WORK', '/kaggle/working/latent-lens')
INPUT = os.environ.get('LL_INPUT', '/kaggle/input')
SIZE = 224
MODEL = 'vit_small_patch14_dinov2.lvd142m'
R, FE, CK, FG = [os.path.join(WORK, s) for s in ('results', 'features', 'ckpt', 'figs')]
for _p in (R, FE, CK, FG):
    os.makedirs(_p, exist_ok=True)
OK, BAD, WARN = '[ok]  ', '[FAIL]', '[warn]'

VARIANTS = {
    'frozen':     dict(train=False),
    'b3':         dict(train=True, nn=False, fn=False, fmix=False),
    'b4':         dict(train=True, nn=True,  fn=True,  fmix=True),
    'b4_no_nn':   dict(train=True, nn=False, fn=True,  fmix=True),
    'b4_no_fmix': dict(train=True, nn=True,  fn=True,  fmix=False),
    'b4_no_fn':   dict(train=True, nn=True,  fn=False, fmix=True),
    'b3_fn':      dict(train=True, nn=False, fn=True,  fmix=False),   # ladder: B3 + false-negative mask only
    'b3_fmix':    dict(train=True, nn=False, fn=False, fmix=True),    # ladder: B3 + Fourier style swap only
}


def say(tag, msg):
    print(f'{tag} {msg}', flush=True)


# ----------------------------------------------------------------------------- data
def find_cache():
    local = os.path.join(WORK, 'data')
    if os.path.exists(os.path.join(local, 'images_224.npy')) and os.path.exists(os.path.join(local, 'meta.csv')):
        return local
    for root, _, names in os.walk(INPUT):
        if 'images_224.npy' in names and 'meta.csv' in names:
            return root
    sys.exit(f'{BAD} images_224.npy / meta.csv not found. In Kaggle: Add Input -> Notebook Output -> your '
             f'"Latent Lens 0" notebook (or run step 0 first).')


def circle_mask(size=SIZE, frac=0.98):
    yy, xx = np.mgrid[:size, :size]
    c = (size - 1) / 2
    return (((yy - c) ** 2 + (xx - c) ** 2) <= (frac * size / 2) ** 2).astype(np.float32)


def l2(Fm):
    return Fm / (np.linalg.norm(Fm, axis=1, keepdims=True) + 1e-8)


def label_cols(rf):
    skip = {'path', 'stem', 'dataset', 'split', 'w', 'h', 'ID', 'size', 'camera', 'camera_source', 'dr_grade',
            'dme_grade', 'mean_r', 'mean_g', 'mean_b', 'std_r', 'std_g', 'std_b', 'sharp', 'retina_w_frac'}
    cols = []
    for c in rf.columns:
        if c in skip:
            continue
        v = rf[c].dropna().unique()
        if len(v) and set(np.unique(v)).issubset({0, 1, 0.0, 1.0}):
            cols.append(c)
    return cols


# ----------------------------------------------------------------------------- probes (same as step 0)
def _pipe(C=1.0):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    return make_pipeline(StandardScaler(), LogisticRegression(C=C, max_iter=3000, class_weight='balanced'))


def camera_leakage(Fm, cams, seed=0):
    from sklearn.model_selection import StratifiedKFold, cross_val_predict
    from sklearn.metrics import accuracy_score, balanced_accuracy_score
    cams = np.asarray(cams)
    vc = pd.Series(cams).value_counts()
    if len(vc) < 2 or vc.min() < 5:
        return None
    pred = cross_val_predict(_pipe(), l2(Fm), cams, cv=StratifiedKFold(5, shuffle=True, random_state=seed))
    return dict(acc=float(accuracy_score(cams, pred)), bal_acc=float(balanced_accuracy_score(cams, pred)),
                majority_rate=float(vc.max() / len(cams)), chance_bal_acc=float(1 / len(vc)), n_cameras=int(len(vc)))


def macro_auc(Ftr, Ytr, Fte, Yte, cols, min_pos_tr, min_pos_te):
    from sklearn.metrics import roc_auc_score
    aucs = {}
    for j, c in enumerate(cols):
        if Ytr[:, j].sum() < min_pos_tr or Yte[:, j].sum() < min_pos_te or (1 - Yte[:, j]).sum() < min_pos_te:
            continue
        clf = _pipe().fit(Ftr, Ytr[:, j])
        aucs[c] = float(roc_auc_score(Yte[:, j], clf.predict_proba(Fte)[:, 1]))
    return dict(macro_auc=float(np.mean(list(aucs.values()))) if aucs else None, n_classes=len(aucs), per_class=aucs)


def few_shot(Fm, meta, cols, cam, shots=(1, 5, 10), n_seeds=10, min_pos=15):
    """K positives + K negatives per class from ONE camera, scored on the rest of that camera.
    Support sets use fixed seeds, so every variant is judged on identical support sets."""
    from sklearn.linear_model import LogisticRegression
    from sklearn.metrics import roc_auc_score
    rf = meta[(meta.dataset == 'rfmid') & (meta.camera == cam)]
    if len(rf) == 0:
        return {}
    Fc = l2(Fm[rf.index.values])
    Yc = rf[cols].fillna(0).values.astype(int)
    elig = [j for j in range(len(cols)) if Yc[:, j].sum() >= min_pos and (1 - Yc[:, j]).sum() >= min_pos]
    out = {}
    for K in shots:
        per_seed = []
        for s in range(n_seeds):
            rng = np.random.RandomState(1000 + s)
            aucs = []
            for j in elig:
                pos, neg = np.where(Yc[:, j] == 1)[0], np.where(Yc[:, j] == 0)[0]
                if len(pos) < K + 5 or len(neg) < K + 5:
                    continue
                sup = np.r_[rng.choice(pos, K, replace=False), rng.choice(neg, K, replace=False)]
                q = np.setdiff1d(np.arange(len(Fc)), sup)
                if Yc[q, j].sum() < 5 or (1 - Yc[q, j]).sum() < 5:
                    continue
                clf = LogisticRegression(C=10, max_iter=2000, class_weight='balanced')
                clf.fit(Fc[sup], np.r_[np.ones(K), np.zeros(K)])
                aucs.append(roc_auc_score(Yc[q, j], clf.decision_function(Fc[q])))
            if aucs:
                per_seed.append(float(np.mean(aucs)))
        if per_seed:
            out[str(K)] = dict(mean=float(np.mean(per_seed)), std=float(np.std(per_seed)),
                               n_classes=len(elig), camera=cam)
    return out


def center_by_camera(Fm, meta):
    """Subtract each camera's own mean embedding (cameras known from image size; no disease labels used).
    Applied identically to every model. For a held-out camera this uses its UNLABELED images (disclose it)."""
    Fn = l2(Fm).copy()
    cams = meta.camera.values
    for c in np.unique(cams):
        m = cams == c
        Fn[m] -= Fn[m].mean(0, keepdims=True)
    return Fn


def evaluate(Fm, meta, hold=None, center=False):
    from sklearn.metrics import roc_auc_score
    if center:
        Fm = center_by_camera(Fm, meta)
    res = {}
    rf_pos = np.where(meta.dataset.values == 'rfmid')[0]
    rf = meta.iloc[rf_pos].reset_index(drop=True)
    cols = label_cols(rf)
    Y = rf[cols].fillna(0).values.astype(int)
    Fr = l2(Fm[rf_pos])
    cams = rf.camera.values
    ok = (cams != hold) if hold else np.ones(len(rf), bool)
    tr = (rf.split == 'train').values & ok
    va = (rf.split == 'val').values & ok
    te = (rf.split == 'test').values & ok
    res['camera_leakage'] = camera_leakage(Fr, cams)
    res['val'] = macro_auc(Fr[tr], Y[tr], Fr[va], Y[va], cols, 30, 5)
    res['test'] = macro_auc(Fr[tr], Y[tr], Fr[te], Y[te], cols, 30, 5)
    if hold:
        h = cams == hold
        res['heldcam'] = macro_auc(Fr[tr], Y[tr], Fr[h], Y[h], cols, 20, 5)
        res['heldcam']['camera'] = hold
    # hidden-set transfer: RFMiD "DR" probe -> IDRiD (any DR = grade>=1, referable = grade>=2)
    idr = meta[meta.dataset == 'idrid']
    if len(idr) and 'DR' in cols and 'dr_grade' in idr.columns and idr.dr_grade.notna().any():
        j = cols.index('DR')
        clf = _pipe().fit(Fr[tr], Y[tr, j])
        s = clf.predict_proba(l2(Fm[idr.index.values]))[:, 1]
        g = idr.dr_grade.values
        res['idrid'] = dict(auc_dr_any=float(roc_auc_score(g >= 1, s)), auc_dr_referable=float(roc_auc_score(g >= 2, s)),
                            n=int(len(idr)), probe_trained_without=hold or 'nothing held out')
    res['few_shot'] = few_shot(Fm, meta, cols, hold or 'cam1')
    return res


# ----------------------------------------------------------------------------- neighbours
def build_nbrs(F0, P, cam_int, n_total, k=5):
    """Top-k most similar training images by FROZEN features: any camera, and a different camera only."""
    Fn = l2(F0[P])
    S = Fn @ Fn.T
    np.fill_diagonal(S, -2.0)
    na = np.argsort(-S, axis=1)[:, :k]
    S2 = S.copy()
    S2[cam_int[P][:, None] == cam_int[P][None, :]] = -2.0
    nc = np.argsort(-S2, axis=1)[:, :k]
    nbr_all = np.full((n_total, k), -1, np.int64)
    nbr_cross = np.full((n_total, k), -1, np.int64)
    nbr_all[P] = P[na]
    nbr_cross[P] = P[nc]
    return nbr_all, nbr_cross


# ----------------------------------------------------------------------------- torch parts
def make_model(dev):
    import timm
    m = timm.create_model(MODEL, pretrained=True, num_classes=0, img_size=SIZE)
    return m.to(dev)


def norm_stats(model, dev):
    import torch, timm
    cfg = timm.data.resolve_data_config({}, model=model)
    mean = torch.tensor(cfg['mean'], device=dev).view(1, 3, 1, 1)
    std = torch.tensor(cfg['std'], device=dev).view(1, 3, 1, 1)
    return mean, std


def extract(model, X, mean, std, mask, dev, bs=128):
    import torch
    model.eval()
    out = []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            xb = torch.from_numpy(np.ascontiguousarray(X[i:i + bs])).to(dev).permute(0, 3, 1, 2).float() / 255.0
            xb = (xb * mask - mean) / std
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(dev == 'cuda')):
                f = model(xb)
            out.append(f.float().cpu().numpy())
    return np.concatenate(out)


def fourier_mix(x, xo, lam, beta=0.1):
    """Swap the low-frequency amplitude (global colour/illumination 'camera style') with another image's."""
    import torch
    fx, fo = torch.fft.fft2(x, dim=(-2, -1)), torch.fft.fft2(xo, dim=(-2, -1))
    ax = torch.fft.fftshift(fx.abs(), dim=(-2, -1))
    ao = torch.fft.fftshift(fo.abs(), dim=(-2, -1))
    ph = fx.angle()
    h = x.shape[-1]
    c, b = h // 2, max(int(beta * h), 1)
    sl = (slice(None), slice(None), slice(c - b, c + b + 1), slice(c - b, c + b + 1))
    ax[sl] = (1 - lam) * ax[sl] + lam * ao[sl]
    ax = torch.fft.ifftshift(ax, dim=(-2, -1))
    out = torch.fft.ifft2(ax * torch.exp(1j * ph)).real
    return out.clamp(0, 1)


def nce(anchor, cands, pos_col, ign, tau):
    import torch.nn.functional as F
    logits = (anchor @ cands.T) / tau
    logits = logits.masked_fill(ign, float('-inf'))
    return F.cross_entropy(logits, pos_col)


def compute_loss(z1, z2, z3, idx, pidx, nbr_all_t, cfg, tau):
    import torch
    B = z1.shape[0]
    ar = torch.arange(B, device=z1.device)

    def mask_for(src, extra=None):
        m = idx[:, None] == src[None, :]                       # same source image (duplicates in a batch)
        if cfg['fn']:                                          # frozen near-duplicates are not negatives
            nb = nbr_all_t[idx]
            m = m | (nb[:, :, None] == src[None, None, :]).any(1)
        if extra is not None:
            m = m | extra
        return m

    ign = mask_for(idx)
    ign[ar, ar] = False
    loss = 0.5 * (nce(z1, z2, ar, ign, tau) + nce(z2, z1, ar, ign, tau))
    if cfg['nn']:
        src = torch.cat([idx, pidx])
        ign2 = mask_for(src, extra=(pidx[:, None] == src[None, :]))
        ign2[ar, ar] = True                                    # the own second view is neither target nor negative
        ign2[ar, B + ar] = False                               # the cross-camera partner view is the target
        loss = loss + nce(z1, torch.cat([z2, z3]), B + ar, ign2, tau)
    return loss


def train(model, X, meta, P, cam_int, nbr_all, nbr_cross, cfg, args, dev, mean, std, mask):
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    from torchvision.transforms import v2
    from torchvision.utils import save_image

    tf = v2.Compose([v2.ToDtype(torch.float32, scale=True),
                     v2.RandomResizedCrop(SIZE, scale=(0.5, 1.0), ratio=(0.9, 1.1), antialias=True),
                     v2.RandomHorizontalFlip(),
                     v2.RandomRotation(20),
                     v2.ColorJitter(0.2, 0.2, 0.1, 0.02),
                     v2.RandomApply([v2.GaussianBlur(5, (0.1, 1.0))], p=0.2)])

    class Views(Dataset):
        def __len__(self):
            return len(P)

        def __getitem__(self, k):
            i = int(P[k])
            p = int(random.choice(list(nbr_cross[i]))) if cfg['nn'] else i
            t = lambda j: torch.from_numpy(np.ascontiguousarray(X[j])).permute(2, 0, 1)
            return tf(t(i)), tf(t(i)), (tf(t(p)) if cfg['nn'] else torch.zeros(1)), i, p

    # camera-balanced sampling (cam0 is 65% of RFMiD)
    cc = cam_int[P]
    w = 1.0 / pd.Series(cc).map(pd.Series(cc).value_counts()).values
    sampler = WeightedRandomSampler(torch.from_numpy(w).double(), num_samples=len(P), replacement=True)
    nw = min(4, os.cpu_count() or 1)
    loader = DataLoader(Views(), batch_size=args.bs, sampler=sampler, num_workers=nw, drop_last=True,
                        persistent_workers=nw > 0)

    # trainable: last K blocks + final norm + projection head
    for p_ in model.parameters():
        p_.requires_grad = False
    train_params = []
    for blk in model.blocks[-args.k_blocks:]:
        train_params += list(blk.parameters())
    train_params += list(model.norm.parameters())
    for p_ in train_params:
        p_.requires_grad = True
    dim = model.num_features
    head = nn.Sequential(nn.Linear(dim, 512), nn.GELU(), nn.Linear(512, 128)).to(dev)
    opt = torch.optim.AdamW([dict(params=train_params, lr=args.lr), dict(params=head.parameters(), lr=args.lr * 10)],
                            weight_decay=0.05)
    base = [g['lr'] for g in opt.param_groups]
    total = args.epochs * len(loader)
    if args.max_steps:
        total = min(total, args.max_steps)
    warm = max(int(0.05 * total), 1)
    use_amp = dev == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    nbr_all_t = torch.from_numpy(nbr_all).to(dev)
    cam_t = torch.from_numpy(cam_int).to(dev)

    say(OK, f'training {args.variant}: {len(P)} images, batch {args.bs}, {len(loader)} steps/epoch, '
            f'{total} steps total, last {args.k_blocks} blocks trainable '
            f'({sum(p.numel() for p in train_params)/1e6:.1f}M params), nn={cfg["nn"]} fn={cfg["fn"]} fmix={cfg["fmix"]}')
    step, t0, losses, saved_fig = 0, time.time(), [], False
    model.train()
    while step < total:
        for a, b, c, idx, pidx in loader:
            if step >= total:
                break
            lr_f = (step + 1) / warm if step < warm else 0.5 * (1 + np.cos(np.pi * (step - warm) / max(total - warm, 1)))
            for g, b0 in zip(opt.param_groups, base):
                g['lr'] = b0 * lr_f
            a, b = a.to(dev), b.to(dev)
            idx, pidx = idx.to(dev), pidx.to(dev)
            if cfg['nn']:
                c = c.to(dev)
                bad = (cam_t[idx] == cam_t[pidx]).sum().item()
                if bad:
                    sys.exit(f'{BAD} {bad} cross-camera partners share the anchor camera (bug)')
            if cfg['fmix']:
                cb = cam_t[idx]
                Rm = torch.rand(len(idx), len(idx), device=dev)
                Rm[cb[:, None] == cb[None, :]] = -1
                v, j = Rm.max(1)
                lam = (torch.rand(len(idx), 1, 1, 1, device=dev) * 0.7 + 0.3) * (v >= 0).float().view(-1, 1, 1, 1)
                a = fourier_mix(a, a[j], lam)
                Rm2 = torch.rand(len(idx), len(idx), device=dev)
                Rm2[cb[:, None] == cb[None, :]] = -1
                b = fourier_mix(b, b[Rm2.argmax(1)], lam)
            if not saved_fig:
                save_image(torch.cat([a[:4], b[:4], c[:4] if cfg['nn'] else a[:4]]).cpu(),
                           os.path.join(FG, f'aug_check_{args.variant}.png'), nrow=4)
                say(OK, f'LOOK AT {FG}/aug_check_{args.variant}.png: row1 view A, row2 view B (colour-style swapped), '
                        f'row3 cross-camera partner. Retinas must stay clearly visible.')
                saved_fig = True
            views = [a, b] + ([c] if cfg['nn'] else [])
            x = torch.cat(views)
            x = (x * mask - mean) / std
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                z = head(model(x))
            z = F.normalize(z.float(), dim=1)
            B = a.shape[0]
            z1, z2 = z[:B], z[B:2 * B]
            z3 = z[2 * B:] if cfg['nn'] else None
            loss = compute_loss(z1, z2, z3, idx, pidx, nbr_all_t, cfg, args.tau)
            if not torch.isfinite(loss):
                sys.exit(f'{BAD} loss is {loss.item()} at step {step}. Lower --lr and retry.')
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.item()))
            step += 1
            if step % 20 == 0 or step == 1 or step == total:
                say(OK, f'step {step}/{total} loss {np.mean(losses[-20:]):.3f} ({time.time()-t0:.0f}s)')
    model.eval()
    torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad},
               os.path.join(CK, f'{args.run}.pt'))
    n_tr = sum(p.numel() for p in train_params)
    return losses, n_tr


# ----------------------------------------------------------------------------- run / report
def run(args):
    import torch
    cfg = VARIANTS[args.variant]
    if args.smoke:
        args.epochs, args.max_steps = 1, 3
    args.run = f'{args.variant}' + (f'_hold-{args.hold}' if args.hold else '') + f'_s{args.seed}' + ('_smoke' if args.smoke else '')
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    say(OK, f'run {args.run} on {dev}')
    if dev == 'cpu' and cfg['train'] and not args.smoke:
        say(WARN, 'no GPU: this will be very slow. Settings -> Accelerator -> GPU.')
    cdir = find_cache()
    X = np.load(os.path.join(cdir, 'images_224.npy'))
    meta = pd.read_csv(os.path.join(cdir, 'meta.csv'))
    assert len(X) == len(meta), f'cache mismatch: {len(X)} images vs {len(meta)} meta rows'
    say(OK, f'cache {cdir}: {X.shape}')
    cams_rf = sorted(c for c in meta.camera.unique() if str(c).startswith(('cam', 'cl')))
    cmap = {c: i for i, c in enumerate(cams_rf)}
    cam_int = meta.camera.map(cmap).fillna(-1).astype(int).values
    if args.hold and args.hold not in cmap:
        sys.exit(f'{BAD} --hold {args.hold} is not a camera. Cameras: {cams_rf}')

    P = np.where((meta.dataset.values == 'rfmid') & (meta.split.values == 'train') & (cam_int >= 0)
                 & (meta.camera.values != args.hold))[0]
    say(OK, 'pretraining images per camera: ' + ', '.join(f'{c}:{int((meta.camera.values[P]==c).sum())}' for c in cams_rf if c != args.hold))
    mask = torch.from_numpy(circle_mask()).to(dev).view(1, 1, SIZE, SIZE)

    model = make_model(dev)
    mean, std = norm_stats(model, dev)
    f0p = os.path.join(FE, 'frozen_masked.npy')
    if os.path.exists(f0p):
        F0 = np.load(f0p)
    else:
        t = time.time()
        F0 = extract(model, X, mean, std, mask, dev)
        np.save(f0p, F0)
        say(OK, f'frozen masked features {F0.shape} ({time.time()-t:.0f}s)')
    res = dict(run=args.run, variant=args.variant, hold=args.hold, seed=args.seed, cfg=cfg,
               params_total=int(sum(p.numel() for p in model.parameters())))
    if cfg['train']:
        use_nn = cfg['nn']
        if len(set(cam_int[P])) < 2 and use_nn:
            sys.exit(f'{BAD} need at least 2 cameras in the training set for cross-camera positives')
        nbr_all, nbr_cross = build_nbrs(F0, P, cam_int, len(meta))
        if use_nn:
            part = nbr_cross[P].ravel()
            same = (cam_int[np.repeat(P, nbr_cross.shape[1])] == cam_int[part]).mean()
            say(OK if same == 0 else BAD, f'cross-camera partners: {100*(1-same):.0f}% have a different camera than their anchor')
        t = time.time()
        losses, n_tr = train(model, X, meta, P, cam_int, nbr_all, nbr_cross, cfg, args, dev, mean, std, mask)
        res.update(train_seconds=time.time() - t, params_trainable=int(n_tr),
                   loss_first=float(np.mean(losses[:5])), loss_last=float(np.mean(losses[-5:])))
        say(OK if res['loss_last'] < res['loss_first'] else WARN, f'loss {res["loss_first"]:.3f} -> {res["loss_last"]:.3f}')
        F1 = extract(model, X, mean, std, mask, dev)
    else:
        F1 = F0
    if not np.isfinite(F1).all():
        sys.exit(f'{BAD} features contain NaN/inf')
    np.save(os.path.join(FE, f'{args.run}.npy'), F1.astype(np.float16))
    t = time.time()
    res.update(evaluate(F1, meta, args.hold))
    say(OK, f'evaluation done ({time.time()-t:.0f}s)')
    json.dump(res, open(os.path.join(R, f'{args.run}.json'), 'w'), indent=1)
    summary(res)


def g(d, *ks):
    for k in ks:
        if not isinstance(d, dict) or k not in d:
            return None
        d = d[k]
    return d


def summary(r):
    f = lambda v: 'n/a' if v is None else f'{v:.3f}'
    say(OK, f'{r["run"]}: camera leakage {f(g(r,"camera_leakage","bal_acc"))} | val AUROC {f(g(r,"val","macro_auc"))} | '
            f'test AUROC {f(g(r,"test","macro_auc"))} | IDRiD DR {f(g(r,"idrid","auc_dr_any"))}' +
            (f' | held-out {r["hold"]} AUROC {f(g(r,"heldcam","macro_auc"))}' if r.get('hold') else ''))
    fs = r.get('few_shot') or {}
    if fs:
        say(OK, 'few-shot AUROC on ' + str(g(fs, '1', 'camera')) + ': ' +
            ', '.join(f'{k}-shot {fs[k]["mean"]:.3f}+-{fs[k]["std"]:.3f}' for k in fs))


RUN_RE = re.compile(r'^(?P<variant>[a-z0-9_]+?)(?:_hold-(?P<hold>cam\d+))?_s(?P<seed>\d+)$')


def collect(sub, ext):
    """Files in WORK/<sub> plus any <sub> folder under INPUT (outputs of earlier notebooks). WORK wins on name clashes."""
    found = {}
    dirs = [os.path.join(WORK, sub)]
    for root, ds, _ in os.walk(INPUT):
        if os.path.basename(root) == sub:
            dirs.append(root)
    for d in dirs[1:] + dirs[:1]:
        for p in glob.glob(os.path.join(d, '*' + ext)):
            found[os.path.basename(p)[:-len(ext)]] = p
    return found


def reeval():
    """Re-score every saved feature file after per-camera mean-centering. No training, no GPU needed."""
    cdir = find_cache()
    meta = pd.read_csv(os.path.join(cdir, 'meta.csv'))
    feats = collect('features', '.npy')
    done = collect('results', '.json')
    n = 0
    for name, p in sorted(feats.items()):
        m = RUN_RE.match(name)
        if not m or '_smoke' in name or name.endswith('_cc') or (name + '_cc') in done:
            continue
        t = time.time()
        F = np.load(p).astype(np.float32)
        if len(F) != len(meta):
            say(WARN, f'{name}: {len(F)} rows != {len(meta)}, skipped')
            continue
        hold = m.group('hold')
        res = dict(run=name + '_cc', variant=m.group('variant'), hold=hold, seed=int(m.group('seed')), centered=True)
        res.update(evaluate(F, meta, hold, center=True))
        json.dump(res, open(os.path.join(R, f'{name}_cc.json'), 'w'), indent=1)
        say(OK, f'{name} (camera-centered, {time.time()-t:.0f}s)')
        summary(res)
        n += 1
    if not n:
        say(WARN, 'nothing new to re-score (already done, or no feature files like b4_hold-cam1_s0.npy found)')


def report(hide_cc=False):
    rows = []
    for name, p in sorted(collect('results', '.json').items()):
        if name == 'poc0' or '_smoke' in name or name == 'summary':
            continue
        r = json.load(open(p))
        if 'variant' not in r or (hide_cc and r.get('centered')):
            continue
        fs = r.get('few_shot') or {}
        rows.append({'variant': r['variant'] + (' +camera-centering' if r.get('centered') else ''),
                     'held out': r.get('hold') or '-', 'seed': r.get('seed', 0),
                     'camera leak': g(r, 'camera_leakage', 'bal_acc'),
                     'val AUROC': g(r, 'val', 'macro_auc'), 'test AUROC': g(r, 'test', 'macro_auc'),
                     'held-cam AUROC': g(r, 'heldcam', 'macro_auc'), 'IDRiD DR AUROC': g(r, 'idrid', 'auc_dr_any'),
                     'IDRiD referable': g(r, 'idrid', 'auc_dr_referable'),
                     '1-shot': g(fs, '1', 'mean'), '5-shot': g(fs, '5', 'mean'), '10-shot': g(fs, '10', 'mean')})
    if not rows:
        sys.exit(f'{BAD} no finished runs in {R}')
    df = pd.DataFrame(rows).sort_values(['held out', 'variant', 'seed'])
    num = [c for c in df.columns if c not in ('variant', 'held out', 'seed')]
    agg = df.groupby(['held out', 'variant'])[num].agg(['mean', 'std'])
    cnt = df.groupby(['held out', 'variant']).size()
    out = []
    for (h, v), row in agg.iterrows():
        d = {'held out': h, 'variant': v, 'seeds': int(cnt[(h, v)])}
        for c in num:
            mu, sd = row[(c, 'mean')], row[(c, 'std')]
            d[c] = 'n/a' if pd.isna(mu) else (f'{mu:.3f}' if pd.isna(sd) else f'{mu:.3f}+-{sd:.3f}')
        out.append(d)
    t1 = pd.DataFrame(out).to_markdown(index=False)
    t2 = df.round(3).to_markdown(index=False)
    txt = ('# Latent Lens runs\n\n## Mean +- std over training seeds (std shown only when seeds > 1)\n\n' + t1 +
           '\n\n## Every run\n\n' + t2 + '\n')
    open(os.path.join(R, 'summary.md'), 'w').write(txt)
    print(t1)
    say(OK, f'wrote {R}/summary.md')


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--stage', default='run', choices=['run', 'report', 'reeval'])
    ap.add_argument('--variant', default='b4', choices=list(VARIANTS))
    ap.add_argument('--hold', default=None, help='camera never used for training or probe fitting, e.g. cam1')
    ap.add_argument('--epochs', type=int, default=20)
    ap.add_argument('--bs', type=int, default=96)
    ap.add_argument('--k_blocks', type=int, default=4)
    ap.add_argument('--lr', type=float, default=1e-4)
    ap.add_argument('--tau', type=float, default=0.2)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--max_steps', type=int, default=0)
    ap.add_argument('--smoke', action='store_true')
    ap.add_argument('--hide_cc', action='store_true', help='report: leave out the camera-centering rows')
    a = ap.parse_args()
    print(f'WORK={WORK}\n')
    (report(a.hide_cc) if a.stage == 'report' else reeval()) if a.stage != 'run' else run(a)


In [ ]:
%%writefile latentlens_trust.py
#!/usr/bin/env python
"""Latent Lens, step 4: trust layer + efficiency numbers. Needs latentlens_train.py in the same folder.

  !python latentlens_trust.py --stage feats    # unfamiliar-image score, calibration (ECE), retrieval; mean +- std over seeds
  !python latentlens_trust.py --stage umap     # UMAP figure coloured by camera and by disease
  !python latentlens_trust.py --stage stress   # blur / low-res / JPEG / dark / low-contrast / noise (needs GPU + the .pt files)
  !python latentlens_trust.py --stage eff      # parameters, FLOPs, CPU/GPU latency, training cost
  !python latentlens_trust.py --stage all      # everything, then writes results/trust_summary.md

Models compared everywhere: frozen DINOv2-S, plain SimCLR (--base, default b3) and Latent Lens (--final, default b3_fmix).
Every number comes from the saved features / checkpoints of real runs. Nothing is tuned on these metrics.
"""
import os, sys, json, time, argparse
import numpy as np
import pandas as pd
import latentlens_train as T
from latentlens_train import say, OK, BAD, WARN, l2

R, FG = T.R, T.FG
NAMES = ['Frozen DINOv2-S', 'Plain SimCLR', 'Latent Lens']


def run_names(args, seed, hold=None):
    h = f'_hold-{hold}' if hold else ''
    return {'Frozen DINOv2-S': 'frozen', 'Plain SimCLR': f'{args.base}{h}_s{seed}', 'Latent Lens': f'{args.final}{h}_s{seed}'}


def load_feats(run):
    name = 'frozen_masked' if run == 'frozen' else run
    p = T.collect('features', '.npy').get(name)
    if p is None:
        sys.exit(f'{BAD} features for {name} not found. Attach the notebook outputs that trained it '
                 f'(and run `--variant {name.split("_hold")[0].rsplit("_s", 1)[0]}` first if it was never trained).')
    return np.load(p).astype(np.float32)


def load_meta():
    return pd.read_csv(os.path.join(T.find_cache(), 'meta.csv'))


def auroc(neg, pos):
    from sklearn.metrics import roc_auc_score
    y = np.r_[np.zeros(len(neg)), np.ones(len(pos))]
    return float(roc_auc_score(y, np.r_[neg, pos]))


# ----------------------------------------------------------------------------- unfamiliar-image score
def unfamiliar(ref, q, k=5):
    """1 - mean cosine similarity to the k nearest reference images. Higher = more unfamiliar."""
    S = l2(q) @ l2(ref).T
    top = -np.partition(-S, k - 1, axis=1)[:, :k]
    return 1.0 - top.mean(1)


def ood_eval(F, meta, hold):
    rf = (meta.dataset.values == 'rfmid')
    cam, split = meta.camera.values, meta.split.values
    seen = rf & (cam != hold) if hold else rf
    ref, fam = F[seen & (split == 'train')], F[seen & (split == 'test')]
    s_fam = unfamiliar(ref, fam)
    out = {}
    idr = meta.dataset.values == 'idrid'
    if idr.any():
        out['idrid_vs_rfmid_test'] = auroc(s_fam, unfamiliar(ref, F[idr]))
    if hold:
        out['unseen_camera_vs_seen_test'] = auroc(s_fam, unfamiliar(ref, F[rf & (cam == hold)]))
    return out


# ----------------------------------------------------------------------------- calibration
def ece_binary(p, y, bins=10):
    edges = np.linspace(0, 1, bins + 1)
    idx = np.clip(np.digitize(p, edges[1:-1]), 0, bins - 1)
    e = 0.0
    for b in range(bins):
        m = idx == b
        if m.any():
            e += m.mean() * abs(y[m].mean() - p[m].mean())
    return float(e)


def calib_eval(F, meta, hold):
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    from sklearn.linear_model import LogisticRegression
    from scipy.optimize import minimize_scalar
    rf = meta[meta.dataset == 'rfmid'].reset_index(drop=True)
    Fr = l2(F[meta.dataset.values == 'rfmid'])
    cols = T.label_cols(rf)
    Y = rf[cols].fillna(0).values.astype(int)
    ok = (rf.camera.values != hold) if hold else np.ones(len(rf), bool)
    tr, va, te = [((rf.split == s).values & ok) for s in ('train', 'val', 'test')]
    sig = lambda z: 1 / (1 + np.exp(-z))
    before, after, temps = [], [], []
    for j, c in enumerate(cols):
        if Y[tr, j].sum() < 30 or Y[va, j].sum() < 5 or Y[te, j].sum() < 5:
            continue
        clf = make_pipeline(StandardScaler(), LogisticRegression(C=1.0, max_iter=3000)).fit(Fr[tr], Y[tr, j])
        zv, zt = clf.decision_function(Fr[va]), clf.decision_function(Fr[te])

        def nll(t):
            p = np.clip(sig(zv / t), 1e-7, 1 - 1e-7)
            return -np.mean(Y[va, j] * np.log(p) + (1 - Y[va, j]) * np.log(1 - p))
        t_opt = float(minimize_scalar(nll, bounds=(0.05, 20), method='bounded').x)
        before.append(ece_binary(sig(zt), Y[te, j]))
        after.append(ece_binary(sig(zt / t_opt), Y[te, j]))
        temps.append(t_opt)
    return dict(ece_before=float(np.mean(before)), ece_after=float(np.mean(after)),
                mean_temperature=float(np.mean(temps)), n_classes=len(before))


# ----------------------------------------------------------------------------- retrieval
def retrieval_eval(F, meta, hold, k=5):
    rf = meta[meta.dataset == 'rfmid'].reset_index(drop=True)
    Fr = l2(F[meta.dataset.values == 'rfmid'])
    cols = [c for c in T.label_cols(rf) if c != 'Disease_Risk']
    Y = rf[cols].fillna(0).values.astype(int)
    cam = np.asarray(rf.camera.values, dtype=object)
    ok = (cam != hold) if hold else np.ones(len(rf), bool)
    gal = (rf.split == 'train').values & ok
    qry = (cam == hold) if hold else (rf.split == 'test').values
    qry = qry & (Y.sum(1) > 0)
    Fq, Fg, Yq, Yg, cq, cg = Fr[qry], Fr[gal], Y[qry], Y[gal], cam[qry], cam[gal]
    match = (Yq @ Yg.T) > 0
    S = Fq @ Fg.T
    same = cq[:, None] == cg[None, :]

    def prec(Sm):
        top = np.argsort(-Sm, axis=1)[:, :k]
        return float(np.take_along_axis(match, top, 1).mean()), top
    out = {'n_queries': int(qry.sum()), 'chance': float(match.mean())}
    if hold:
        out['precision_at_k_unseen_camera'], _ = prec(S)
    else:
        out['precision_at_k_all'], top = prec(S)
        out['same_camera_share_top_k'] = float(np.take_along_axis(same, top, 1).mean())
        out['same_camera_share_base_rate'] = float(same.mean())
        out['precision_at_k_cross_camera'], _ = prec(np.where(same, -np.inf, S))
    return out


# ----------------------------------------------------------------------------- aggregate seeds
def msd(vals):
    v = np.array([x for x in vals if x is not None], float)
    return None if not len(v) else (float(v.mean()), float(v.std()))


def feats_stage(args):
    meta = load_meta()
    res = {}
    for name in NAMES:
        per = {}
        for hold in (None, args.hold):
            tag = 'hold' if hold else 'seen'
            runs = []
            for s in args.seeds:
                F = load_feats(run_names(args, s, hold)[name])
                runs.append({'ood': ood_eval(F, meta, hold), 'calib': calib_eval(F, meta, hold),
                             'retr': retrieval_eval(F, meta, hold)})
            per[tag] = runs
            say(OK, f'{name} [{tag}] done for seeds {args.seeds}')
        res[name] = per
    json.dump(res, open(os.path.join(R, 'trust_feats.json'), 'w'), indent=1)
    say(OK, f'wrote {R}/trust_feats.json')
    return res


# ----------------------------------------------------------------------------- UMAP figure
def umap_stage(args):
    import matplotlib
    matplotlib.use('Agg')
    import matplotlib.pyplot as plt
    meta = load_meta()
    rng = np.random.RandomState(0)
    idx = np.sort(rng.choice(len(meta), min(1800, len(meta)), replace=False))
    sub = meta.iloc[idx]
    cams = sub.camera.values
    dis = sub['Disease_Risk'].values if 'Disease_Risk' in sub.columns else np.full(len(sub), np.nan)
    try:
        import umap
        reducer = lambda: umap.UMAP(n_neighbors=15, min_dist=0.1, metric='cosine', random_state=0)
        method = 'UMAP'
    except ImportError:
        from sklearn.decomposition import PCA
        reducer = lambda: PCA(2, random_state=0)
        method = 'PCA (umap-learn not installed)'
        say(WARN, 'umap-learn missing: using PCA. In Kaggle run: !pip -q install umap-learn')
    plt.rcParams.update({'font.size': 9, 'axes.spines.top': False, 'axes.spines.right': False})
    fig, ax = plt.subplots(2, 3, figsize=(10.5, 6.4), dpi=150)
    camcol = {'cam0': '#0072B2', 'cam1': '#E69F00', 'cam2': '#009E73', 'idrid': '#CC79A7'}
    rn = run_names(args, 0)
    for j, name in enumerate(NAMES):
        Z = reducer().fit_transform(l2(load_feats(rn[name])[idx]))
        for c, col in camcol.items():
            m = cams == c
            if m.any():
                ax[0, j].scatter(Z[m, 0], Z[m, 1], s=4, c=col, alpha=0.7, linewidths=0, rasterized=True,
                                 label=('IDRiD' if c == 'idrid' else c))
        lab = np.isnan(dis.astype(float))
        ax[1, j].scatter(Z[lab, 0], Z[lab, 1], s=4, c='#DDDDDD', alpha=0.6, linewidths=0, rasterized=True, label='IDRiD (no label)')
        for v, col, nm in ((0, '#0072B2', 'no disease flag'), (1, '#D55E00', 'disease risk')):
            m = (~lab) & (dis.astype(float) == v)
            ax[1, j].scatter(Z[m, 0], Z[m, 1], s=4, c=col, alpha=0.6, linewidths=0, rasterized=True, label=nm)
        ax[0, j].set_title(name, fontsize=10, fontweight='bold')
        for i in (0, 1):
            ax[i, j].set_xticks([]); ax[i, j].set_yticks([])
    for i, lbl in enumerate(('(a) coloured by camera', '(b) coloured by disease label')):
        ax[i, 0].set_ylabel(lbl, fontsize=9)
        h, l = ax[i, 2].get_legend_handles_labels()
        ax[i, 2].legend(h, l, markerscale=3, frameon=False, fontsize=8, loc='upper left', bbox_to_anchor=(1.02, 1.0))
    fig.suptitle(f'{method} of image embeddings, coloured by camera (top) and by disease label (bottom)', fontsize=11, y=0.995)
    fig.tight_layout()
    for ext in ('png', 'pdf'):
        fig.savefig(os.path.join(FG, f'umap_camera_vs_disease.{ext}'), dpi=300, bbox_inches='tight')
    say(OK, f'LOOK AT {FG}/umap_camera_vs_disease.png ({method})')


# ----------------------------------------------------------------------------- stress test (torch)
def degrade(X, kind):
    import cv2
    out = np.empty_like(X)
    rng = np.random.RandomState(0)
    for i, x in enumerate(X):
        if kind == 'blur_sigma2':
            y = cv2.GaussianBlur(x, (0, 0), 2)
        elif kind == 'blur_sigma4':
            y = cv2.GaussianBlur(x, (0, 0), 4)
        elif kind == 'low_res_4x':
            y = cv2.resize(cv2.resize(x, (56, 56), interpolation=cv2.INTER_AREA), (224, 224), interpolation=cv2.INTER_LINEAR)
        elif kind == 'jpeg_q20':
            _, b = cv2.imencode('.jpg', x[:, :, ::-1], [cv2.IMWRITE_JPEG_QUALITY, 20])
            y = cv2.imdecode(b, 1)[:, :, ::-1]
        elif kind == 'dark_x0.4':
            y = np.clip(x.astype(np.float32) * 0.4, 0, 255)
        elif kind == 'low_contrast_x0.5':
            m = x.reshape(-1, 3).mean(0)
            y = np.clip(m + 0.5 * (x.astype(np.float32) - m), 0, 255)
        elif kind == 'noise_sd15':
            y = np.clip(x.astype(np.float32) + rng.normal(0, 15, x.shape), 0, 255)
        else:
            raise ValueError(kind)
        out[i] = np.asarray(y).astype(np.uint8)
    return out


KINDS = ['blur_sigma2', 'blur_sigma4', 'low_res_4x', 'jpeg_q20', 'dark_x0.4', 'low_contrast_x0.5', 'noise_sd15']


def stress_stage(args):
    import torch
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    cdir = T.find_cache()
    X = np.load(os.path.join(cdir, 'images_224.npy'))
    meta = pd.read_csv(os.path.join(cdir, 'meta.csv'))
    rf = meta[meta.dataset == 'rfmid']
    cols = T.label_cols(rf)
    Y = rf[cols].fillna(0).values.astype(int)
    sp = rf.split.values
    tr, te = sp == 'train', sp == 'test'
    rf_pos = np.where(meta.dataset.values == 'rfmid')[0]
    te_pos = rf_pos[te]
    Xte = X[te_pos]
    mask = torch.from_numpy(T.circle_mask()).to(dev).view(1, 1, T.SIZE, T.SIZE)
    ckpts = T.collect('ckpt', '.pt')
    rn = run_names(args, 0)
    res = {}
    for name in NAMES:
        model = T.make_model(dev)
        mean, std = T.norm_stats(model, dev)
        if rn[name] != 'frozen':
            if rn[name] not in ckpts:
                sys.exit(f'{BAD} checkpoint {rn[name]}.pt not found. Attach the notebook output that trained it.')
            r = model.load_state_dict(torch.load(ckpts[rn[name]], map_location='cpu'), strict=False)
            if r.unexpected_keys:
                sys.exit(f'{BAD} unexpected keys in checkpoint: {r.unexpected_keys[:3]}')
        Ffile = load_feats(rn[name])
        Ftr = l2(Ffile[rf_pos[tr]])
        clean = T.extract(model, Xte, mean, std, mask, dev)
        diff = float(np.abs(l2(clean) - l2(Ffile[te_pos])).max())
        say(OK if diff < 0.05 else BAD, f'{name}: recomputed clean features match the saved ones (max abs diff {diff:.4f})')
        ref = Ftr
        rows = {}
        base = T.macro_auc(Ftr, Y[tr], l2(clean), Y[te], cols, 30, 5)['macro_auc']
        rows['clean'] = dict(macro_auc=base, unfamiliar_auroc=None)
        s_clean = unfamiliar(ref, clean)
        for kind in KINDS:
            Fd = T.extract(model, degrade(Xte, kind), mean, std, mask, dev)
            rows[kind] = dict(macro_auc=T.macro_auc(Ftr, Y[tr], l2(Fd), Y[te], cols, 30, 5)['macro_auc'],
                              unfamiliar_auroc=auroc(s_clean, unfamiliar(ref, Fd)))
            say(OK, f'{name:16s} {kind:18s} AUROC {rows[kind]["macro_auc"]:.3f} (clean {base:.3f}) | '
                    f'degraded-vs-clean unfamiliar AUROC {rows[kind]["unfamiliar_auroc"]:.3f}')
        res[name] = rows
        del model
    json.dump(res, open(os.path.join(R, 'trust_stress.json'), 'w'), indent=1)
    say(OK, f'wrote {R}/trust_stress.json')


# ----------------------------------------------------------------------------- efficiency (torch)
def eff_stage(args):
    import torch, timm
    from torch.utils.flop_counter import FlopCounterMode
    rows = {}
    makers = {'DINOv2-S backbone (Latent Lens, frozen DINOv2-S, SimCLR)': lambda: T.make_model('cpu'),
              'ResNet50 (reference only)': lambda: timm.create_model('resnet50.a1_in1k', pretrained=False, num_classes=0)}
    for name, mk in makers.items():
        m = mk().eval()
        x = torch.randn(1, 3, T.SIZE, T.SIZE)
        with torch.no_grad():
            with FlopCounterMode(display=False) as fc:
                m(x)
            for _ in range(3):
                m(x)
            ts = []
            for _ in range(20):
                t = time.perf_counter(); m(x); ts.append(time.perf_counter() - t)
        r = dict(params_m=sum(p.numel() for p in m.parameters()) / 1e6, gflops=fc.get_total_flops() / 1e9,
                 cpu_ms_batch1=float(np.median(ts) * 1e3), cpu_threads=torch.get_num_threads())
        if torch.cuda.is_available():
            mg, xg = m.cuda(), x.cuda()
            with torch.no_grad():
                for _ in range(10):
                    mg(xg)
                torch.cuda.synchronize()
                ts = []
                for _ in range(50):
                    t = time.perf_counter(); mg(xg); torch.cuda.synchronize(); ts.append(time.perf_counter() - t)
            r['gpu_ms_batch1'] = float(np.median(ts) * 1e3)
            r['gpu'] = torch.cuda.get_device_name(0)
        rows[name] = r
        say(OK, f'{name}: {r["params_m"]:.1f}M params, {r["gflops"]:.1f} GFLOPs, CPU {r["cpu_ms_batch1"]:.0f} ms/image')
    ck = T.collect('ckpt', '.pt').get(f'{args.final}_s0')
    if ck:
        sd = torch.load(ck, map_location='cpu')
        rows['trainable_params_m_latent_lens'] = sum(v.numel() for v in sd.values()) / 1e6
    secs = []
    for n, p in T.collect('results', '.json').items():
        if n.startswith(args.final + '_s') and '_hold' not in n:
            secs.append(json.load(open(p)).get('train_seconds'))
    secs = [s for s in secs if s]
    if secs:
        rows['train_seconds_median'] = float(np.median(secs))
        rows['train_runs'] = len(secs)
    json.dump(rows, open(os.path.join(R, 'trust_eff.json'), 'w'), indent=1)
    say(OK, f'wrote {R}/trust_eff.json')


# ----------------------------------------------------------------------------- summary
def fmt(v):
    return 'n/a' if v is None else f'{v[0]:.3f}' + (f'+-{v[1]:.3f}' if v[1] > 0 else '')


def summary_stage(args):
    L = ['# Latent Lens: trust layer and efficiency', '']
    p = os.path.join(R, 'trust_feats.json')
    if os.path.exists(p):
        d = json.load(open(p))
        g = lambda name, tag, sec, key: msd([r[sec].get(key) for r in d[name][tag]])
        L += ['## Unfamiliar-image score (kNN distance in embedding space), AUROC, higher = better', '',
              '| model | IDRiD vs RFMiD test (all cameras seen) | unseen camera cam1 vs seen test (cam1 held out) |', '|:--|--:|--:|']
        for n in NAMES:
            L.append(f'| {n} | {fmt(g(n,"seen","ood","idrid_vs_rfmid_test"))} | {fmt(g(n,"hold","ood","unseen_camera_vs_seen_test"))} |')
        L += ['', '## Calibration of per-class probes (ECE on test, lower = better; temperature fitted on validation)', '',
              '| model | ECE before | ECE after temperature scaling |', '|:--|--:|--:|']
        for n in NAMES:
            L.append(f'| {n} | {fmt(g(n,"seen","calib","ece_before"))} | {fmt(g(n,"seen","calib","ece_after"))} |')
        L += ['', '## Retrieval: do the 5 nearest training images share a disease with the query? (precision@5)', '',
              '| model | all gallery | cross-camera gallery only | same-camera share of top-5 (base rate ' +
              (f'{d[NAMES[0]]["seen"][0]["retr"]["same_camera_share_base_rate"]:.2f}' if d else 'n/a') + ') | unseen camera cam1 queries |',
              '|:--|--:|--:|--:|--:|']
        for n in NAMES:
            L.append(f'| {n} | {fmt(g(n,"seen","retr","precision_at_k_all"))} | {fmt(g(n,"seen","retr","precision_at_k_cross_camera"))} | '
                     f'{fmt(g(n,"seen","retr","same_camera_share_top_k"))} | {fmt(g(n,"hold","retr","precision_at_k_unseen_camera"))} |')
    p = os.path.join(R, 'trust_stress.json')
    if os.path.exists(p):
        d = json.load(open(p))
        L += ['', '## Robustness to image-quality shifts (RFMiD test, macro AUROC; probes trained on clean images)', '',
              '| condition | ' + ' | '.join(NAMES) + ' |', '|:--|' + '--:|' * len(NAMES)]
        for k in ['clean'] + KINDS:
            L.append(f'| {k} | ' + ' | '.join(f'{d[n][k]["macro_auc"]:.3f}' for n in NAMES) + ' |')
        L += ['', 'Degraded-vs-clean detection with the unfamiliar-image score (AUROC):', '',
              '| condition | ' + ' | '.join(NAMES) + ' |', '|:--|' + '--:|' * len(NAMES)]
        for k in KINDS:
            L.append(f'| {k} | ' + ' | '.join(f'{d[n][k]["unfamiliar_auroc"]:.3f}' for n in NAMES) + ' |')
    p = os.path.join(R, 'trust_eff.json')
    if os.path.exists(p):
        d = json.load(open(p))
        L += ['', '## Efficiency (224x224, one image)', '', '| model | params (M) | GFLOPs | CPU ms | GPU ms |', '|:--|--:|--:|--:|--:|']
        for k, v in d.items():
            if isinstance(v, dict):
                L.append(f'| {k} | {v["params_m"]:.1f} | {v["gflops"]:.1f} | {v["cpu_ms_batch1"]:.0f} ({v["cpu_threads"]} threads) | '
                         f'{v.get("gpu_ms_batch1", float("nan")):.1f} |')
        if 'trainable_params_m_latent_lens' in d:
            L.append(f'\nTrainable during fine-tuning: {d["trainable_params_m_latent_lens"]:.1f}M parameters.')
        if 'train_seconds_median' in d:
            L.append(f'Training time per run (median of {d["train_runs"]} runs): {d["train_seconds_median"]/60:.1f} min on the Kaggle GPU.')
    open(os.path.join(R, 'trust_summary.md'), 'w').write('\n'.join(L) + '\n')
    print('\n'.join(L))
    say(OK, f'wrote {R}/trust_summary.md')


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--stage', default='all', choices=['all', 'feats', 'umap', 'stress', 'eff', 'summary'])
    ap.add_argument('--base', default='b3')
    ap.add_argument('--final', default='b3_fmix')
    ap.add_argument('--hold', default='cam1')
    ap.add_argument('--seeds', type=int, nargs='+', default=[0, 1, 2])
    a = ap.parse_args()
    print(f'WORK={T.WORK}\n')
    if a.stage in ('all', 'feats'):
        feats_stage(a)
    if a.stage in ('all', 'umap'):
        umap_stage(a)
    if a.stage in ('all', 'stress'):
        stress_stage(a)
    if a.stage in ('all', 'eff'):
        eff_stage(a)
    if a.stage in ('all', 'summary'):
        summary_stage(a)


In [ ]:
%%writefile latentlens_step5.py
#!/usr/bin/env python
"""Latent Lens, step 5: rare-disease evidence, better unfamiliar score, supervised baseline, camera hold-out summary.
Needs latentlens_train.py and latentlens_trust.py in the same folder.

  !python latentlens_step5.py --stage sup --seed 0         # supervised baseline (labels of the common classes only), GPU
  !python latentlens_step5.py --stage rare                 # rare vs common diseases, few-shot on rare ones, paired confidence intervals
  !python latentlens_step5.py --stage ood2                 # unfamiliar score: kNN distance vs Mahalanobis distance
  !python latentlens_step5.py --stage loco                 # table for every held-out camera (cam0, cam1, cam2)

Definitions, fixed before looking at any result:
  common class = at least 30 positive training images (this is what every earlier table used);
  rare class   = 8 to 29 positive training images AND at least 4 positives in val+test together.
  Rare classes are scored on val+test pooled (too few positives in test alone). Probes are fit on train only.
Every number comes from saved features of real runs.
"""
import os, sys, json, time, random, argparse
import numpy as np
import pandas as pd
import latentlens_train as T
import latentlens_trust as U
from latentlens_train import say, OK, BAD, WARN, l2

R, FE, CK = T.R, T.FE, T.CK
MODELS = [('Frozen DINOv2-S', 'frozen'), ('Plain SimCLR', 'b3'), ('Latent Lens', 'b3_fmix'), ('Supervised fine-tune', 'sup')]
SHOTS = (3, 5, 10)


def fast_auc(y, s):
    from scipy.stats import rankdata
    y = np.asarray(y).astype(bool)
    n1, n0 = int(y.sum()), int((~y).sum())
    if n1 == 0 or n0 == 0:
        return np.nan
    r = rankdata(s)
    return float((r[y].sum() - n1 * (n1 + 1) / 2) / (n1 * n0))


def setup(args):
    meta = U.load_meta()
    rf_mask = meta.dataset.values == 'rfmid'
    rf = meta[rf_mask].reset_index(drop=True)
    cols = T.label_cols(rf)
    Y = rf[cols].fillna(0).values.astype(int)
    have = T.collect('features', '.npy')
    feats = {}
    for name, var in MODELS:
        seeds = [0] if var == 'frozen' else args.seeds
        key = lambda s: 'frozen_masked' if var == 'frozen' else f'{var}_s{s}'
        if not all(key(s) in have for s in seeds):
            if var == 'sup':
                say(WARN, 'no supervised-baseline features found: that row is left out (run --stage sup first)')
                continue
            sys.exit(f'{BAD} features missing for {name}: need {[key(s) for s in seeds]}')
        feats[name] = [l2(np.load(have[key(s)]).astype(np.float32)[rf_mask]) for s in seeds]
    return meta, rf, cols, Y, feats


def probe_scores(Ftr, ytr, Fev):
    clf = T._pipe().fit(Ftr, ytr)
    return clf.predict_proba(Fev)[:, 1]


def msd(v):
    v = np.asarray([x for x in v if x is not None and not np.isnan(x)], float)
    return (float(v.mean()), float(v.std())) if len(v) else (np.nan, 0.0)


def fmt(m, s=None):
    if m is None or (isinstance(m, float) and np.isnan(m)):
        return 'n/a'
    return f'{m:.3f}' + (f'+-{s:.3f}' if s else '')


# ----------------------------------------------------------------------------- rare vs common
def rare_stage(args):
    meta, rf, cols, Y, feats = setup(args)
    sp = rf.split.values
    tr, te, pool = sp == 'train', sp == 'test', sp != 'train'
    ntr = Y[tr].sum(0)
    common = [j for j in range(len(cols)) if ntr[j] >= 30 and Y[te, j].sum() >= 5 and (1 - Y[te, j]).sum() >= 5]
    rare = [j for j in range(len(cols)) if 8 <= ntr[j] < 30 and Y[pool, j].sum() >= 4]
    dropped = [(cols[j], int(ntr[j])) for j in range(len(cols)) if j not in common and j not in rare and ntr[j] > 0]
    say(OK, f'{len(common)} common classes, {len(rare)} rare classes: ' + ', '.join(f'{cols[j]}({ntr[j]})' for j in rare))
    say(OK, f'{len(dropped)} classes too small to evaluate (left out, disclosed): ' + ', '.join(f'{c}({n})' for c, n in dropped))
    names = list(feats)
    rng = np.random.RandomState(0)

    # scores: sc[name][seed_idx][class_pos] -> scores over the eval set
    def all_scores(classes, ev):
        sc = {}
        for n in names:
            sc[n] = []
            for Fm in feats[n]:
                sc[n].append([probe_scores(Fm[tr], Y[tr, j], Fm[ev]) for j in classes])
        return sc
    t = time.time()
    sc_rare = all_scores(rare, pool)
    sc_com = all_scores(common, te)
    say(OK, f'probes fitted ({time.time()-t:.0f}s)')

    def table(sc, classes, ev):
        yv = np.stack([Y[ev, j] for j in classes])
        per = {n: np.array([[fast_auc(yv[k], sc[n][s][k]) for k in range(len(classes))] for s in range(len(sc[n]))]) for n in names}
        return yv, per
    yv_r, per_r = table(sc_rare, rare, pool)
    yv_c, per_c = table(sc_com, common, te)

    # paired bootstrap over evaluation images (same resample for every model)
    def boot(sc, yv, B):
        N = yv.shape[1]
        out = {n: [] for n in names}
        for _ in range(B):
            idx = rng.randint(0, N, N)
            ys = yv[:, idx]
            valid = [k for k in range(len(ys)) if 0 < ys[k].sum() < N]
            if not valid:
                continue
            for n in names:
                v = [np.nanmean([fast_auc(ys[k], sc[n][s][k][idx]) for k in valid]) for s in range(len(sc[n]))]
                out[n].append(np.mean(v))
        return {n: np.array(v) for n, v in out.items()}
    t = time.time()
    b_r, b_c = boot(sc_rare, yv_r, args.boot), boot(sc_com, yv_c, args.boot)
    say(OK, f'bootstrap done ({time.time()-t:.0f}s)')

    def ci(a, b):
        d = b_arr[b] - b_arr[a]
        return float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)), float((d > 0).mean())

    L = ['# Latent Lens step 5: rare vs common diseases', '',
         f'Common classes (>=30 train positives, scored on test): {len(common)}. Rare classes (8-29 train positives, scored on val+test pooled): {len(rare)}.',
         'Classes with fewer than 8 training positives cannot be evaluated and are left out: ' + ', '.join(f'{c} ({n})' for c, n in dropped) + '.', '']
    res = {'common': [cols[j] for j in common], 'rare': [cols[j] for j in rare], 'left_out': dropped}
    for title, per, bt in (('RARE classes', per_r, b_r), ('COMMON classes', per_c, b_c)):
        L += [f'## {title}: macro AUROC, linear probe fit on all training images (mean +- std over training seeds)', '',
              '| model | macro AUROC | 95% bootstrap interval of the mean over resampled eval images |', '|:--|--:|--:|']
        for n in names:
            m = per[n].mean(1)
            lo, hi = np.percentile(bt[n], [2.5, 97.5])
            L.append(f'| {n} | {fmt(*msd(m))} | {lo:.3f} to {hi:.3f} |')
            res.setdefault(title, {})[n] = dict(mean=msd(m)[0], std=msd(m)[1], ci=[float(lo), float(hi)])
        b_arr = bt
        L += ['', 'Paired differences (same resampled images for both models; "P(better)" = share of resamples where the first model wins):', '',
              '| comparison | mean difference | 95% interval | P(better) |', '|:--|--:|--:|--:|']
        for a, b in (('Frozen DINOv2-S', 'Latent Lens'), ('Plain SimCLR', 'Latent Lens'), ('Frozen DINOv2-S', 'Plain SimCLR'),
                     ('Supervised fine-tune', 'Latent Lens')):
            if a in names and b in names:
                d, lo, hi, p = ci(a, b)
                L.append(f'| {b} minus {a} | {d:+.3f} | {lo:+.3f} to {hi:+.3f} | {p:.2f} |')
                res.setdefault(title + '_paired', {})[f'{b} - {a}'] = dict(diff=d, lo=lo, hi=hi, p_better=p)
        L.append('')
    L += ['## Per rare class (AUROC, mean over training seeds)', '',
          '| class | train positives | val+test positives | ' + ' | '.join(names) + ' |', '|:--|--:|--:|' + '--:|' * len(names)]
    for k, j in enumerate(rare):
        L.append(f'| {cols[j]} | {int(ntr[j])} | {int(Y[pool, j].sum())} | ' + ' | '.join(f'{per_r[n][:, k].mean():.3f}' for n in names) + ' |')

    # few-shot on the rare classes: K positives + K negatives from the TRAIN images, scored on val+test
    from sklearn.linear_model import LogisticRegression
    L += ['', '## Few-shot on rare classes (K positive + K negative training images, scored on val+test; 10 fixed support sets, identical for every model)', '',
          '| K | ' + ' | '.join(names) + ' | classes |', '|--:|' + '--:|' * len(names) + '--:|']
    fs_res = {}
    for K in SHOTS:
        classes = [j for j in rare if ntr[j] >= K]
        grid = {n: np.full((len(feats[n]), args.support_sets, len(classes)), np.nan) for n in names}
        for s in range(args.support_sets):
            for kk, j in enumerate(classes):
                r = np.random.RandomState(1000 + s * 100 + j)
                pos, neg = np.where(tr & (Y[:, j] == 1))[0], np.where(tr & (Y[:, j] == 0))[0]
                sup = np.r_[r.choice(pos, K, replace=False), r.choice(neg, K, replace=False)]
                for n in names:
                    for si, Fm in enumerate(feats[n]):
                        clf = LogisticRegression(C=10, max_iter=2000, class_weight='balanced').fit(Fm[sup], np.r_[np.ones(K), np.zeros(K)])
                        grid[n][si, s, kk] = fast_auc(Y[pool, j], clf.decision_function(Fm[pool]))
        row = []
        for n in names:
            per_set = np.nanmean(grid[n], axis=(0, 2))              # one number per support set (mean over seeds, classes)
            row.append(fmt(per_set.mean(), per_set.std()))
        L.append(f'| {K} | ' + ' | '.join(row) + f' | {len(classes)} |')
        fs_res[str(K)] = {n: float(np.nanmean(grid[n])) for n in names}
        if 'Frozen DINOv2-S' in names and 'Latent Lens' in names:
            d = np.nanmean(grid['Latent Lens'], axis=0) - np.nanmean(grid['Frozen DINOv2-S'], axis=0)   # support set x class
            cl = np.nanmean(d, axis=0)
            bs = [np.mean(cl[rng.randint(0, len(cl), len(cl))]) for _ in range(2000)]
            fs_res[str(K)]['latent_minus_frozen'] = dict(mean=float(np.nanmean(d)), win_rate=float((d > 0).mean()),
                                                        ci_over_classes=[float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))])
    L += ['', 'Latent Lens minus frozen, per (class, support set) pair:', '', '| K | mean difference | share of pairs where Latent Lens is better | 95% interval (resampling classes) |', '|--:|--:|--:|--:|']
    for K in SHOTS:
        d = fs_res[str(K)].get('latent_minus_frozen')
        if d:
            L.append(f'| {K} | {d["mean"]:+.3f} | {d["win_rate"]:.2f} | {d["ci_over_classes"][0]:+.3f} to {d["ci_over_classes"][1]:+.3f} |')
    res['few_shot_rare'] = fs_res

    # the nearly empty image seen in the augmentation check
    sd = [c for c in ('std_r', 'std_g', 'std_b') if c in meta.columns]
    if sd:
        m = meta.assign(std_mean=meta[sd].mean(1)).sort_values('std_mean')
        L += ['', '## Lowest-contrast images in the data (check for near-empty pictures)', '', '| row | dataset | split | camera | mean pixel std |', '|--:|:--|:--|:--|--:|']
        for i, r in m.head(8).iterrows():
            L.append(f'| {i} | {r["dataset"]} | {r["split"]} | {r["camera"]} | {r["std_mean"]:.1f} |')
    open(os.path.join(R, 'rare_summary.md'), 'w').write('\n'.join(L) + '\n')
    json.dump(res, open(os.path.join(R, 'rare.json'), 'w'), indent=1, default=float)
    print('\n'.join(L))
    say(OK, f'wrote {R}/rare_summary.md and rare.json')


# ----------------------------------------------------------------------------- unfamiliar score, second method
def ood2_stage(args):
    from sklearn.covariance import LedoitWolf
    meta = U.load_meta()
    rfm = meta.dataset.values == 'rfmid'
    cam, split = meta.camera.values, meta.split.values
    idr = meta.dataset.values == 'idrid'

    def maha(ref, q):
        lw = LedoitWolf().fit(ref)
        d = q - lw.location_
        return np.einsum('ij,jk,ik->i', d, lw.precision_, d)

    def one(F, hold):
        F = l2(F)
        seen = rfm & (cam != hold) if hold else rfm
        ref, fam = F[seen & (split == 'train')], F[seen & (split == 'test')]
        out = {}
        for nm, fn in (('knn', lambda r, q: U.unfamiliar(r, q)), ('mahalanobis', maha)):
            s0 = fn(ref, fam)
            if idr.any():
                out[f'{nm}_idrid'] = U.auroc(s0, fn(ref, F[idr]))
            if hold:
                out[f'{nm}_unseen_camera'] = U.auroc(s0, fn(ref, F[rfm & (cam == hold)]))
        return out
    res, rows = {}, []
    for name, var in MODELS[:3]:
        for hold in (None, args.hold):
            seeds = [0] if var == 'frozen' else args.seeds
            vals = []
            for s in seeds:
                run = 'frozen' if var == 'frozen' else f'{var}{"_hold-" + hold if hold else ""}_s{s}'
                vals.append(one(U.load_feats(run), hold))
            res[f'{name} [{"hold " + hold if hold else "all seen"}]'] = vals
            say(OK, f'{name} [{hold or "seen"}] done')
    L = ['# Unfamiliar-image score: kNN distance vs Mahalanobis distance (AUROC, higher = better; 0.5 = chance)', '',
         '| model | kNN: IDRiD vs RFMiD test | Mahalanobis: IDRiD vs RFMiD test | kNN: unseen camera cam1 | Mahalanobis: unseen camera cam1 |', '|:--|--:|--:|--:|--:|']
    for name, _ in MODELS[:3]:
        a, b = res[f'{name} [all seen]'], res[f'{name} [hold {args.hold}]']
        g = lambda lst, k: fmt(*msd([x.get(k) for x in lst]))
        L.append(f'| {name} | {g(a, "knn_idrid")} | {g(a, "mahalanobis_idrid")} | {g(b, "knn_unseen_camera")} | {g(b, "mahalanobis_unseen_camera")} |')
    open(os.path.join(R, 'ood2_summary.md'), 'w').write('\n'.join(L) + '\n')
    json.dump(res, open(os.path.join(R, 'ood2.json'), 'w'), indent=1, default=float)
    print('\n'.join(L))
    say(OK, f'wrote {R}/ood2_summary.md')


# ----------------------------------------------------------------------------- every held-out camera
def loco_stage(args):
    done = T.collect('results', '.json')
    L = ['# Held-out camera: frozen vs plain SimCLR vs Latent Lens (macro AUROC on the camera that was never used for training or probe fitting)', '',
         '| held-out camera | model | seeds | held-camera AUROC | 5-shot on that camera | IDRiD DR AUROC | RFMiD test AUROC |', '|:--|:--|--:|--:|--:|--:|--:|']
    per = {}
    for hold in ('cam0', 'cam1', 'cam2'):
        for name, var in MODELS:
            runs = [json.load(open(p)) for n, p in done.items() if n.startswith(f'{var}_hold-{hold}_s')]
            if not runs:
                continue
            g = T.g
            h = [g(r, 'heldcam', 'macro_auc') for r in runs]
            f5 = [g(r, 'few_shot', '5', 'mean') for r in runs]
            idr = [g(r, 'idrid', 'auc_dr_any') for r in runs]
            te = [g(r, 'test', 'macro_auc') for r in runs]
            per[(hold, name)] = h
            L.append(f'| {hold} | {name} | {len(runs)} | {fmt(*msd(h))} | {fmt(*msd(f5))} | {fmt(*msd(idr))} | {fmt(*msd(te))} |')
    L += ['', '## Latent Lens versus the others, per seed (held-camera AUROC difference)', '', '| held-out camera | comparison | per-seed differences | seeds where Latent Lens is better |', '|:--|:--|:--|--:|']
    for hold in ('cam0', 'cam1', 'cam2'):
        for other in ('Frozen DINOv2-S', 'Plain SimCLR'):
            a, b = per.get((hold, 'Latent Lens')), per.get((hold, other))
            if a and b:
                if len(b) == 1:
                    b = b * len(a)
                d = [x - y for x, y in zip(a, b) if x is not None and y is not None]
                L.append(f'| {hold} | Latent Lens minus {other} | ' + ', '.join(f'{x:+.3f}' for x in d) + f' | {sum(x > 0 for x in d)}/{len(d)} |')
    open(os.path.join(R, 'loco_summary.md'), 'w').write('\n'.join(L) + '\n')
    print('\n'.join(L))
    say(OK, f'wrote {R}/loco_summary.md')


# ----------------------------------------------------------------------------- supervised baseline (torch)
def sup_stage(args):
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
    from torchvision.transforms import v2
    if args.smoke:
        args.epochs, args.max_steps = 1, 3
    run = 'sup' + (f'_hold-{args.hold}' if args.hold else '') + f'_s{args.seed}' + ('_smoke' if args.smoke else '')
    random.seed(args.seed); np.random.seed(args.seed); torch.manual_seed(args.seed)
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    say(OK, f'run {run} on {dev}')
    cdir = T.find_cache()
    X = np.load(os.path.join(cdir, 'images_224.npy'))
    meta = pd.read_csv(os.path.join(cdir, 'meta.csv'))
    rf_all = meta[meta.dataset == 'rfmid']
    cols = T.label_cols(rf_all.reset_index(drop=True))
    cams = sorted(c for c in meta.camera.unique() if str(c).startswith(('cam', 'cl')))
    cmap = {c: i for i, c in enumerate(cams)}
    cam_int = meta.camera.map(cmap).fillna(-1).astype(int).values
    P = np.where((meta.dataset.values == 'rfmid') & (meta.split.values == 'train') & (cam_int >= 0) & (meta.camera.values != args.hold))[0]
    Yall = meta[cols].fillna(0).values.astype(np.float32)
    sup_cols = [j for j in range(len(cols)) if Yall[P, j].sum() >= 30]
    say(OK, f'supervised on {len(P)} train images, {len(sup_cols)} common classes (rare classes are NOT supervised)')
    Ys = Yall[:, sup_cols]
    pw = torch.from_numpy(np.clip((len(P) - Ys[P].sum(0)) / np.maximum(Ys[P].sum(0), 1), 1, 10)).float().to(dev)
    mask = torch.from_numpy(T.circle_mask()).to(dev).view(1, 1, T.SIZE, T.SIZE)
    model = T.make_model(dev)
    mean, std = T.norm_stats(model, dev)
    tf = v2.Compose([v2.ToDtype(torch.float32, scale=True), v2.RandomResizedCrop(T.SIZE, scale=(0.5, 1.0), ratio=(0.9, 1.1), antialias=True),
                     v2.RandomHorizontalFlip(), v2.RandomRotation(20), v2.ColorJitter(0.2, 0.2, 0.1, 0.02)])

    class DS(Dataset):
        def __len__(self):
            return len(P)

        def __getitem__(self, k):
            i = int(P[k])
            return tf(torch.from_numpy(np.ascontiguousarray(X[i])).permute(2, 0, 1)), torch.from_numpy(Ys[i])
    cc = cam_int[P]
    w = 1.0 / pd.Series(cc).map(pd.Series(cc).value_counts()).values
    nw = min(4, os.cpu_count() or 1)
    loader = DataLoader(DS(), batch_size=args.bs, sampler=WeightedRandomSampler(torch.from_numpy(w).double(), len(P), replacement=True),
                        num_workers=nw, drop_last=True, persistent_workers=nw > 0)
    for p_ in model.parameters():
        p_.requires_grad = False
    tp = []
    for blk in model.blocks[-args.k_blocks:]:
        tp += list(blk.parameters())
    tp += list(model.norm.parameters())
    for p_ in tp:
        p_.requires_grad = True
    head = nn.Linear(model.num_features, len(sup_cols)).to(dev)
    opt = torch.optim.AdamW([dict(params=tp, lr=args.lr), dict(params=head.parameters(), lr=args.lr * 10)], weight_decay=0.05)
    base = [g_['lr'] for g_ in opt.param_groups]
    total = args.epochs * len(loader)
    if args.max_steps:
        total = min(total, args.max_steps)
    warm = max(int(0.05 * total), 1)
    use_amp = dev == 'cuda'
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    step, t0, losses = 0, time.time(), []
    model.train()
    while step < total:
        for xb, yb in loader:
            if step >= total:
                break
            lr_f = (step + 1) / warm if step < warm else 0.5 * (1 + np.cos(np.pi * (step - warm) / max(total - warm, 1)))
            for g_, b0 in zip(opt.param_groups, base):
                g_['lr'] = b0 * lr_f
            xb, yb = xb.to(dev), yb.to(dev)
            with torch.autocast(device_type='cuda', dtype=torch.float16, enabled=use_amp):
                out = head(model((xb * mask - mean) / std))
            loss = F.binary_cross_entropy_with_logits(out.float(), yb, pos_weight=pw)
            if not torch.isfinite(loss):
                sys.exit(f'{BAD} loss is {loss.item()} at step {step}')
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt)
            scaler.update()
            losses.append(float(loss.item()))
            step += 1
            if step % 40 == 0 or step == 1 or step == total:
                say(OK, f'step {step}/{total} loss {np.mean(losses[-20:]):.3f} ({time.time()-t0:.0f}s)')
    model.eval()
    torch.save({n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}, os.path.join(CK, f'{run}.pt'))
    F1 = T.extract(model, X, mean, std, mask, dev)
    if not np.isfinite(F1).all():
        sys.exit(f'{BAD} features contain NaN/inf')
    np.save(os.path.join(FE, f'{run}.npy'), F1.astype(np.float16))
    res = dict(run=run, variant='sup', hold=args.hold, seed=args.seed, params_trainable=int(sum(p.numel() for p in tp)),
               train_seconds=time.time() - t0, loss_first=float(np.mean(losses[:5])), loss_last=float(np.mean(losses[-5:])))
    res.update(T.evaluate(F1, meta, args.hold))
    json.dump(res, open(os.path.join(R, f'{run}.json'), 'w'), indent=1)
    T.summary(res)


if __name__ == '__main__':
    ap = argparse.ArgumentParser()
    ap.add_argument('--stage', required=True, choices=['sup', 'rare', 'ood2', 'loco'])
    ap.add_argument('--seeds', type=int, nargs='+', default=[0, 1, 2])
    ap.add_argument('--hold', default=None)
    ap.add_argument('--seed', type=int, default=0)
    ap.add_argument('--boot', type=int, default=500)
    ap.add_argument('--support_sets', type=int, default=10)
    ap.add_argument('--epochs', type=int, default=20)
    ap.add_argument('--bs', type=int, default=96)
    ap.add_argument('--k_blocks', type=int, default=4)
    ap.add_argument('--lr', type=float, default=1e-4)
    ap.add_argument('--max_steps', type=int, default=0)
    ap.add_argument('--smoke', action='store_true')
    a = ap.parse_args()
    print(f'WORK={T.WORK}\n')
    if a.stage == 'ood2' and not a.hold:
        a.hold = 'cam1'
    {'sup': sup_stage, 'rare': rare_stage, 'ood2': ood2_stage, 'loco': loco_stage}[a.stage](a)


In [ ]:
import os
assert os.path.exists('latentlens_step5.py') and os.path.exists('latentlens_train.py'), 'script files missing'
print('script files present')

In [ ]:
# 1-minute smoke test (3 steps) before the real runs
!python latentlens_step5.py --stage sup --smoke

In [ ]:
!for h in cam1 cam2; do for s in 0 1 2; do python latentlens_step5.py --stage sup --hold $h --seed $s; done; done

In [ ]:
!for s in 0 1 2; do python latentlens_step5.py --stage sup --hold cam0 --epochs 100 --seed $s; done

In [ ]:
!python latentlens_step5.py --stage loco

**Download from Output:** results/loco_summary.md and every results/sup_hold-*.json. Then paste me the log.